# RAG Evaluation with Ragas → Phoenix

Runs the live RAG agent from `initial_rag.ipynb`, collects (question / answer / contexts) per query, evaluates with **ragas** using Vertex AI as the LLM judge, and uploads the scored results to Phoenix as a Dataset (visible in the Phoenix UI at `http://localhost:6006`).

| Group | Metrics | Active when |
|---|---|---|
| **A — Reference-free** | `ResponseRelevancy`, `Faithfulness` | Always |
| **B — Ground-truth** | `ContextPrecision`, `ContextRecall`, `ContextEntityRecall`, `NoiseSensitivity` | `USE_GROUND_TRUTH = True` |

In [ ]:
# # Install dependencies (run once)
# %pip install ragas langchain-google-vertexai --quiet


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────

# Flip to True to also run Group B metrics (requires ground_truth.json)
USE_GROUND_TRUTH: bool = False
GROUND_TRUTH_FILE: str = "ground_truth.json"   # relative to this notebook

# Datastore (same as initial_rag.ipynb)
TARGET_DATASTORE_ID: str = "masked-data_1781603945004"
DATASTORE_LOCATION: str  = "us"

# Phoenix server
PHOENIX_URL: str = "http://localhost:6006"

# Questions to evaluate — add or replace as needed
EVAL_QUESTIONS: list = [
    "tell me about the gaps in facilities",
    "what are the HVAC requirements for the receiving site?",
    "what environmental controls are needed for compound dispensing?",
]

print(f"USE_GROUND_TRUTH : {USE_GROUND_TRUTH}")
print(f"Eval questions   : {len(EVAL_QUESTIONS)}")


In [ ]:
import os
import google.auth
import vertexai
from phoenix.otel import register
from phoenix.client import Client
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

_, project_id = google.auth.default()

# Vertex AI
vertexai.init(project=project_id, location="us-central1")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"]      = project_id
os.environ["GOOGLE_CLOUD_LOCATION"]     = "us-central1"

# Phoenix tracing (ADK spans → Phoenix)
tracer_provider = register(
    project_name="gap_assessment",
    endpoint=f"{PHOENIX_URL}/v1/traces",
)
GoogleADKInstrumentor().instrument(tracer_provider=tracer_provider)

# Phoenix client (for uploading evaluation dataset)
phoenix_client = Client(base_url=PHOENIX_URL)

print(f"Project : {project_id}")
print(f"Phoenix : {PHOENIX_URL}")


In [ ]:
# ── RAG agent code  ───────────────────────────────────
# Modified: make_search_tool_eval accepts a `captured_contexts` list that gets
# populated with retrieved passage text so ragas can use it for evaluation.
# Query expansion is disabled — searches use the original query only.

import json
from typing import Any, Dict, List

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine_v1 as discoveryengine
from google.genai import types as genai_types


CONVERSATIONAL_PROMPT = """\
You are a TS/MS (Technical Services/Manufacturing Science) Lead Scientist supporting a drug product technology transfer of Multiple-dose [COMPOUND_1] Injection (3 mL cartridge) from the Sending Site to a Receiving Site, for Registration Stability / Process Validation batches, as described in the technology transfer gap assessment report (LQP-230-1).

A collection of technical documents is available to you, including: Gap Assessment / Risk Assessment reports (per LQP-230-1), Global Process Flow Documents (GLO_gPFD), Technical Study Summary Reports (VPHP evaluation, semi-dried residues downtime), Qualification Strategies (visual inspection, APS), and site-specific assessments.

IMPORTANT - Tool Use:
Call search_datastore exactly once, passing the user's question as the query. The tool internally handles query expansion and retrieves all relevant passages in a single call. After receiving the results, compile your complete analysis and produce a single, final response. Do not call search_datastore multiple times.

Your Task:
- Identify all facility-related requirements documented at the Sending Site and from global standards (EU Annex 1, GQS202, LQP-230-1).
- Identify all facility-related evidence available from the Receiving Site.
- Analyse requirements and evidence, compare them, and prepare the gaps.

Ground Rules:
- Be precise. Cite the source document and section for each requirement, evidence entry, and gap.
- Do not fabricate gaps not present in the documents.\
"""


def _chunk_search_passages(
    query: str,
    project_id: str,
    location: str,
    datastore_id: str,
    page_size: int = 8,
) -> List[Dict[str, Any]]:
    endpoint = (
        "discoveryengine.googleapis.com"
        if location.lower() == "global"
        else f"{location}-discoveryengine.googleapis.com"
    )
    serving_config = (
        f"projects/{project_id}/locations/{location}/collections/default_collection/"
        f"dataStores/{datastore_id}/servingConfigs/default_config"
    )
    spec = discoveryengine.SearchRequest.ContentSearchSpec(
        search_result_mode=discoveryengine.SearchRequest.ContentSearchSpec.SearchResultMode.CHUNKS,
    )
    req = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=page_size,
        content_search_spec=spec,
    )
    passages: List[Dict[str, Any]] = []
    client = discoveryengine.SearchServiceClient(
        client_options=ClientOptions(api_endpoint=endpoint)
    )
    for r in client.search(req).results:
        chunk = getattr(r, "chunk", None)
        if chunk is None:
            continue
        content = str(getattr(chunk, "content", "") or "").strip()
        if not content:
            continue
        doc_meta = getattr(chunk, "document_metadata", None)
        title    = str(getattr(doc_meta, "title", "") or "") if doc_meta else ""
        uri      = str(getattr(doc_meta, "uri",   "") or "") if doc_meta else ""
        chunk_name = str(getattr(chunk, "name", "") or "")
        parts  = chunk_name.split("/")
        doc_id = parts[-3] if len(parts) >= 3 else ""
        passages.append({"doc_id": doc_id, "title": title, "uri": uri, "content": content})
    return passages


def make_search_tool_eval(
    project_id: str,
    location: str,
    datastore_id: str,
    captured_contexts: List[str],
):
    """ADK-compatible search tool (no query expansion) that captures passages for ragas."""

    def search_datastore(query: str) -> str:
        """Search the document datastore for relevant chunks matching the query.

        Args:
            query: The search query to find relevant document chunks.

        Returns:
            Formatted document chunks with citation IDs, titles, URIs, and content.
        """
        all_passages = _chunk_search_passages(query, project_id, location, datastore_id)

        if not all_passages:
            return "No relevant documents found."

        # Capture passage text for ragas evaluation
        captured_contexts.extend(p["content"] for p in all_passages)

        lines = []
        for idx, p in enumerate(all_passages, 1):
            lines.append(
                f"[C{idx}]\ntitle: {p['title']}\nuri: {p['uri']}\ncontent: {p['content']}"
            )
        return "\n\n".join(lines)

    return search_datastore


async def run_rag_agent_eval(
    query: str,
    user_id: str,
    project_id: str,
    location: str,
    datastore_id: str,
    captured_contexts: List[str],
    model_location: str = "us-central1",
) -> str:
    """Run the RAG agent and populate captured_contexts with retrieved passages."""
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
    os.environ["GOOGLE_CLOUD_PROJECT"]      = project_id
    os.environ["GOOGLE_CLOUD_LOCATION"]     = model_location

    search_tool = make_search_tool_eval(project_id, location, datastore_id, captured_contexts)

    agent = Agent(
        model="gemini-2.5-flash",
        name="rag_agent",
        instruction=CONVERSATIONAL_PROMPT,
        tools=[search_tool],
    )

    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name="rag_eval", user_id=user_id)
    runner  = Runner(agent=agent, app_name="rag_eval", session_service=session_service)

    result_text = ""
    user_message = genai_types.Content(
        role="user",
        parts=[genai_types.Part.from_text(text=query)],
    )
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=user_message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    result_text = part.text
                    break

    return result_text

print("RAG agent code loaded (query expansion disabled).")


In [ ]:
# ── Run evaluation queries through the live RAG agent ─────────────────────────
# Each query → answer + list of retrieved context passages (captured by the tool)

eval_rows = []  # [{question, answer, contexts}, ...]

for i, question in enumerate(EVAL_QUESTIONS, 1):
    print(f"[{i}/{len(EVAL_QUESTIONS)}] Running: {question[:70]}...")
    captured_contexts: List[str] = []

    answer = await run_rag_agent_eval(
        query=question,
        user_id=f"eval-user-{i}",
        project_id=project_id,
        location=DATASTORE_LOCATION,
        datastore_id=TARGET_DATASTORE_ID,
        captured_contexts=captured_contexts,
    )

    eval_rows.append({
        "question": question,
        "answer": answer,
        "contexts": captured_contexts,
    })
    print(f"  → {len(captured_contexts)} contexts retrieved, answer length: {len(answer)} chars\n")

print(f"Done. Collected {len(eval_rows)} eval rows.")


In [ ]:
# ── Metric groups ─────────────────────────────────────────────────────────────

from ragas.metrics import ResponseRelevancy, Faithfulness

# Group A — Reference-free (always active)
group_a_metrics = [
    ResponseRelevancy(),
    Faithfulness(),
]

# Group B — Requires ground truth (only when USE_GROUND_TRUTH = True)
if USE_GROUND_TRUTH:
    from ragas.metrics import (
        ContextPrecision,
        ContextRecall,
        ContextEntityRecall,
        NoiseSensitivity,
    )
    group_b_metrics = [
        ContextPrecision(),
        ContextRecall(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ]
else:
    group_b_metrics = []

metrics = group_a_metrics + group_b_metrics

# ── LLM for ragas judge — Vertex AI via LangChain (uses ADC, no API key) ──────
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings

ragas_llm = LangchainLLMWrapper(
    ChatVertexAI(model_name="gemini-2.5-flash", project=project_id, location="us-central1")
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    VertexAIEmbeddings(model_name="text-embedding-004", project=project_id, location="us-central1")
)

for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ragas_embeddings

print(f"Active metrics ({len(metrics)}):")
for m in metrics:
    print(f"  - {m.name}")


In [ ]:
# ── Load ground truth and merge into eval_rows (only when USE_GROUND_TRUTH=True)

if USE_GROUND_TRUTH:
    gt_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), GROUND_TRUTH_FILE)
    with open(gt_path, "r", encoding="utf-8") as f:
        gt_records = json.load(f)

    gt_lookup = {rec["question"]: rec["ground_truth"] for rec in gt_records}

    missing = [row["question"] for row in eval_rows if row["question"] not in gt_lookup]
    if missing:
        raise ValueError(
            f"Ground truth missing for {len(missing)} question(s):\n"
            + "\n".join(f"  - {q}" for q in missing)
        )

    for row in eval_rows:
        row["ground_truth"] = gt_lookup[row["question"]]

    print(f"Ground truth loaded from '{GROUND_TRUTH_FILE}' and merged into eval rows.")
else:
    print("USE_GROUND_TRUTH=False — skipping ground truth loading.")


In [ ]:
# ── Build EvaluationDataset and run ragas ─────────────────────────────────────

from ragas import EvaluationDataset, SingleTurnSample
from ragas import evaluate

samples = []
for row in eval_rows:
    kwargs = dict(
        user_input=row["question"],
        response=row["answer"],
        retrieved_contexts=row["contexts"],
    )
    if USE_GROUND_TRUTH:
        kwargs["reference"] = row["ground_truth"]
    samples.append(SingleTurnSample(**kwargs))

dataset = EvaluationDataset(samples=samples)

print("Running ragas evaluation...")
ragas_results = evaluate(dataset=dataset, metrics=metrics)
print("Done.")
print(ragas_results)


In [ ]:
# ── Upload results to Phoenix as a Dataset (visible in Phoenix UI) ─────────────
# Navigate to http://localhost:6006 → Datasets → ragas-evaluation

import pandas as pd

df = ragas_results.to_pandas()

# Identify ragas score columns
score_cols = [c for c in df.columns if c not in ("user_input", "response", "retrieved_contexts", "reference")]

phoenix_dataset = phoenix_client.datasets.create_dataset(
    name="ragas-evaluation",
    dataframe=df,
    input_keys=["user_input"],
    output_keys=["response"],
    metadata_keys=score_cols,
)

print(f"Dataset uploaded to Phoenix!")
print(f"  Name    : ragas-evaluation")
print(f"  Rows    : {len(df)}")
print(f"  Metrics : {score_cols}")
print(f"\nView at: {PHOENIX_URL} → Datasets → ragas-evaluation")


In [ ]:
# ── Display results in notebook ───────────────────────────────────────────────

display(df[["user_input"] + score_cols])
